# Session 5 — RAG evaluation: how do you know it works?

**Goal**: measure your RAG system quantitatively, at both layers:
1. **Retrieval quality** — is the right chunk even being fetched? (Hit Rate, MRR)
2. **Answer quality** — is the generated answer faithful and relevant? (LLM-as-judge)

**Why this matters**: retrieval quality caps answer quality. If the retriever fails, no prompt can save you. So we evaluate bottom-up.

**Prerequisites**: everything from sessions 1–4. No new packages — we build the metrics ourselves so you understand them (frameworks like RAGAS automate this later).

With your statistics background: Hit Rate is just accuracy@k, and MRR is a rank-based statistic. You already know these — today you apply them to retrieval.

## 1. Load the vector store

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings,
    collection_name="pdf_docs",
)
print(f"Vectors: {db._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors: 715


## 2. Build a small evaluation dataset

The foundation of any eval: **(question, expected evidence)** pairs. In industry this is called a *golden dataset*.

Open your PDF, pick 8-10 facts you can locate, and write a question for each plus a short *keyphrase* that must appear in a correctly retrieved chunk. Below is a starter set for the Data Analytics Lens — **adapt the keyphrases to what's actually in your chunks** (verify manually first!).

In [9]:
# Each item: the question we'll ask + a keyphrase that the correct chunk must contain.
# The keyphrase acts as our ground-truth relevance label.
eval_set = [
    {"question": "What are the four common data classification levels?",
     "keyphrase": "restricted"},
    {"question": "Which AWS service is used to define and enforce data lake access controls?",
     "keyphrase": "Lake Formation"},
    {"question": "What service logs and monitors account activity related to data access?",
     "keyphrase": "CloudTrail"},
    {"question": "How should you encrypt data at rest?",
     "keyphrase": "KMS"},
    {"question": "What should you use to test application logic with consistent results?",
     "keyphrase": "curated dataset"},
    {"question": "Which service defines CI/CD processes for analytics jobs?",
     "keyphrase": "CodePipeline"},
    # Add 3-4 more from YOUR reading of the PDF
    {"question": "Which service is used to run analytics on real-time data?",
     "keyphrase": "Amazon MSK"},
    {"question": "What must be defined in order to do a correct data discovery?",
     "keyphrase": "Business value"},
    {"question": "In a modern data architecture, which role has interest in data pipeline, data processing, data integration, data governance, and data catalogs?",
     "keyphrase": "Data architect"}    
]
print(f"{len(eval_set)} eval questions ready")

9 eval questions ready


## 3. Retrieval metrics: Hit Rate and MRR

- **Hit Rate@k**: fraction of questions where at least one of the top-k retrieved chunks contains the evidence. (Accuracy of the retriever.)
- **MRR (Mean Reciprocal Rank)**: average of 1/rank of the *first* relevant chunk. Rewards putting the right chunk at position 1, not just somewhere in top-k.

Two systems can have the same Hit Rate but very different MRR — and rank matters because chunks earlier in the prompt get more attention from the LLM.

In [10]:
def evaluate_retrieval(db, eval_set, k=3):
    hits, reciprocal_ranks = 0, []
    details = []

    for item in eval_set:
        docs = db.similarity_search(item["question"], k=k)
        rank = None
        for i, d in enumerate(docs, start=1):
            if item["keyphrase"].lower() in d.page_content.lower():
                rank = i
                break
        if rank:
            hits += 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
        details.append({"q": item["question"][:50], "rank": rank})

    hit_rate = hits / len(eval_set)
    mrr = sum(reciprocal_ranks) / len(eval_set)
    return hit_rate, mrr, details

hit_rate, mrr, details = evaluate_retrieval(db, eval_set, k=3)

print(f"Hit Rate@3: {hit_rate:.2%}")
print(f"MRR:        {mrr:.3f}\n")
for d in details:
    status = f"rank {d['rank']}" if d['rank'] else "MISS"
    print(f"  [{status:>6}] {d['q']}")

Hit Rate@3: 66.67%
MRR:        0.593

  [rank 1] What are the four common data classification level
  [rank 1] Which AWS service is used to define and enforce da
  [rank 3] What service logs and monitors account activity re
  [  MISS] How should you encrypt data at rest?
  [  MISS] What should you use to test application logic with
  [rank 1] Which service defines CI/CD processes for analytic
  [rank 1] Which service is used to run analytics on real-tim
  [  MISS] What must be defined in order to do a correct data
  [rank 1] In a modern data architecture, which role has inte


### Experiment: retrieval parameters

Now you can **tune with numbers instead of vibes**. Re-run the eval varying k:

In [11]:
for k in [1, 3, 5, 10]:
    hr, mrr, _ = evaluate_retrieval(db, eval_set, k=k)
    print(f"k={k:>2}  |  Hit Rate: {hr:.2%}  |  MRR: {mrr:.3f}")

# Note: MRR barely changes with k (it only counts the FIRST relevant hit),
# while Hit Rate can only go up. The gap between them tells you whether
# relevant chunks are ranked high (good) or buried deep (bad).

k= 1  |  Hit Rate: 55.56%  |  MRR: 0.556
k= 3  |  Hit Rate: 66.67%  |  MRR: 0.593
k= 5  |  Hit Rate: 77.78%  |  MRR: 0.615
k=10  |  Hit Rate: 88.89%  |  MRR: 0.633


## 4. Answer quality: LLM-as-judge

Retrieval metrics don't tell you if the *final answer* is good. For that, the standard approach is **LLM-as-judge**: a model grades the answer against the retrieved context.

We'll score **faithfulness**: is every claim in the answer supported by the context? This is THE anti-hallucination metric.

⚠️ Caveat you should say out loud in interviews: using a small local model as judge is noisy — in production you'd use a strong model as judge, keep a human-labeled calibration set, and check judge-human agreement.

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model="llama3.2", temperature=0)
retriever = db.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# The RAG chain from session 3
rag_prompt = ChatPromptTemplate.from_template("""Answer based only on the context. If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

In [13]:
judge_prompt = ChatPromptTemplate.from_template("""You are an evaluator. Given a context and an answer, score the FAITHFULNESS of the answer: are its claims supported by the context?

Reply with EXACTLY one line in this format:
SCORE: <1-5> | REASON: <max 15 words>

Scale: 5 = fully supported, 3 = partially supported, 1 = contradicts or invents facts.

Context:
{context}

Answer to evaluate:
{answer}""")

judge_chain = judge_prompt | llm | StrOutputParser()

def evaluate_answer(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    answer = rag_chain.invoke(question)
    verdict = judge_chain.invoke({"context": context, "answer": answer})
    return answer, verdict

# Run the judge over the eval set
for item in eval_set[:4]:   # first 4 to keep it fast
    answer, verdict = evaluate_answer(item["question"])
    print(f"Q: {item['question']}")
    print(f"A: {answer[:120]}...")
    print(f"JUDGE -> {verdict}\n")

Q: What are the four common data classification levels?
A: According to the context, the four common data classification levels mentioned are:

1. Restricted
2. Confidential
3. In...
JUDGE -> SCORE: 5 | REASON: Claims exactly match the context-provided levels without variation or contradiction.

Q: Which AWS service is used to define and enforce data lake access controls?
A: Based on the context, it appears that AWS Lake Formation is used to define and enforce data lake access controls. This i...
JUDGE -> SCORE: 2 | REASON: Answer partially supports claims about AWS Lake Formation's role in defining and enforcing data lake access controls.

Q: What service logs and monitors account activity related to data access?
A: CloudWatch....
JUDGE -> SCORE: 2 | REASON: Logs are used in troubleshooting, but CloudWatch is not explicitly mentioned as a log storage solution.

Q: How should you encrypt data at rest?
A: According to the context, it seems that data at rest can be encrypted using IAM po

## 5. The adversarial test

The most important row in any RAG eval: questions the documents CANNOT answer. A perfect system says \"I don't know\" — anything else is hallucination, and the judge should catch it.

In [14]:
adversarial = [
    "What is the exact pricing of Redshift Serverless per RPU-hour?",
    "Who won the World Cup in 2010?",
]

for q in adversarial:
    answer, verdict = evaluate_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer[:150]}")
    print(f"JUDGE -> {verdict}\n")

Q: What is the exact pricing of Redshift Serverless per RPU-hour?
A: I don't know. The provided context mentions that with Amazon Redshift Serverless, you pay for compute only when the data warehouse is in use, but it d
JUDGE -> SCORE: 3 | REASON: Answer partially supported by context regarding usage-based pricing.

Q: Who won the World Cup in 2010?
A: I don't know. The context appears to be related to choosing compute, storage, and file solutions, but it doesn't mention sports or the World Cup.
JUDGE -> SCORE: 3 | REASON: Answer partially supports context with correct inference about solution choices.



## Bonus

Re-ingest with `chunk_size=300` into a new collection (`pdf_docs_small`) and run the same eval against both collections.